In [89]:
import pandas as pd 
import numpy as np

In [90]:
df=pd.read_csv("50_Startups.csv")

In [91]:
df.head()

,R&D Spend,Administration,Marketing Spend,State,Profit
0,165349.20,136897.80,471784.10,New York,192261.83
1,162597.70,151377.59,443898.53,California,191792.06
2,153441.51,101145.55,407934.54,Florida,191050.39
3,144372.41,118671.85,383199.62,New York,182901.99
4,142107.34,91391.77,366168.42,Florida,166187.94


In [92]:

X=df.iloc[:,:-1]
Y=df.iloc[:,-1]

In [93]:
categorical_features=['State']
numeric_features=["R&D Spend", "Administration","Marketing Spend"]

In [94]:
#Column transformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
preprocessor=ColumnTransformer(
    transformers=[("categories",OneHotEncoder(drop="first"),categorical_features)],
    remainder='passthrough'
)

In [95]:
from sklearn.pipeline import Pipeline
model=Pipeline([ ('preprocessor',preprocessor), ("regressor",LinearRegression())])

In [96]:
from sklearn.model_selection import train_test_split

X_train,X_test, Y_train, Y_test= train_test_split(X, Y, test_size=0.2, random_state=42)

In [97]:
from sklearn.linear_model import LinearRegression
model.fit(X_train,Y_train)

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\compose\_column_transformer.py:1623: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('categories',
                                                  OneHotEncoder(drop='first'),
                                                  ['State'])])),
                ('regressor', LinearRegression())])

In [98]:
y_pred = model.predict(X_test)
print(y_pred)

[126362.87908252  84608.45383643  99677.49425155  46357.46068582
 128750.48288497  50912.41741905 109741.350327   100643.24281644
  97599.275746   113097.42524437]


In [99]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(Y_test, y_pred)
mse = mean_squared_error(Y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(Y_test, y_pred)

print("Mean Absolute Error:", mae)
print("Mean Squared Error:", mse)
print("Root Mean Squared Error:", rmse)
print("R² Score:", r2)


Mean Absolute Error: 6961.47781327555
Mean Squared Error: 82010363.04501335
Root Mean Squared Error: 9055.957323497796
R² Score: 0.8987266414319841


In [103]:
# hyper parameter tuning
from sklearn.linear_model import Ridge, Lasso
parameter={
    'regressor':[Ridge(),Lasso()],
    'regressor__alpha':[0.01,0.1,1,10,100],
    'regressor__fit_intercept':[True,False]
}


In [104]:

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Ridge())
])
from sklearn.model_selection import GridSearchCV
grid=GridSearchCV(pipeline,param_grid=parameter,cv=5, scoring='r2')


In [105]:
grid.fit(X_train, Y_train)

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\compose\_column_transformer.py:1623: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('categories',
                                                                         OneHotEncoder(drop='first'),
                                                                         ['State'])])),
                                       ('regressor', Ridge())]),
             param_grid={'regressor': [Ridge(), Lasso()],
                         'regressor__alpha': [0.01, 0.1, 1, 10, 100],
                         'regressor__fit_intercept': [True, False]},
             scoring='r2')

In [106]:
print("Best Parameters:", grid.best_params_)
print("Best CV Score (R²):", grid.best_score_)

Best Parameters: {'regressor': Ridge(), 'regressor__alpha': 100, 'regressor__fit_intercept': True}
Best CV Score (R²): 0.9394364805963195


In [107]:
import joblib
joblib.dump(grid,"Linaer_model.pkl")

['Linaer_model.pkl']